# Sprint 4 - FieldCare project notebook

This shared Colab is the technical backbone for the Sprint 4 FieldCare project. You will keep one notebook copy as you scope, specify, build, integrate, test, improve, and assess a more capable AI application.

FieldCare supports field-service technicians who diagnose equipment issues. The application you build should combine service documentation, retrieval, reranking, structured equipment and ticket data, tool results, context assembly, model-ready responses, abstention, and escalation behavior.

Choose **File > Save a copy in Drive** before editing in Colab.

**Resource type:** Campus educational notebook: one Sprint 4 project environment used across all Campus lessons and live sessions.


## 1. Install helper core and notebook dependencies

Run this first in Colab. The helper core is installed from the course GitHub repository so model clients and retrieval helpers match earlier notebooks. The core project cells below also run with deterministic local helpers so you can make progress before enabling any live model call.


In [ ]:
#@title Install helper core from GitHub
%pip install -q --force-reinstall --no-cache-dir "ms-ai-ml-helper-core @ git+https://github.com/richhiey/ai-app-dev_Mod-A.git@main"
%pip install -q "pandas>=2,<3"


## 2. Imports, optional model key, and asset loading

Most Sprint 4 work is deterministic: you inspect data, design the contract, run local retrieval, execute simulated tools, and compare expected behavior. The optional model call uses OpenRouter. Store `OPENROUTER_API_KEY` in Colab Secrets when available. Do not paste an API key into notebook source.


In [ ]:
#@title Hidden setup: imports and asset readers { display-mode: "form" }
from __future__ import annotations

import csv
import json
import math
import os
import re
from collections import Counter
from dataclasses import dataclass
from getpass import getpass
from io import StringIO
from pathlib import Path
from typing import Any, Optional
from urllib.error import URLError
from urllib.request import Request, urlopen

import pandas as pd
from IPython.display import JSON, Markdown, display

try:
    from openrouter import OpenRouterClient
except Exception:
    OpenRouterClient = None

try:
    from documents import Document
    from keyword_search import BM25Retriever
    from tools import ToolRegistry
except Exception as exc:
    Document = None
    BM25Retriever = None
    ToolRegistry = None
    HELPER_CORE_IMPORT_ERROR = repr(exc)
else:
    HELPER_CORE_IMPORT_ERROR = None

SIMULATED_CURRENT_DATE = "2026-09-09"
FIELDCARE_RAW_BASE_URL = "https://raw.githubusercontent.com/richhiey/ai-app-dev_Mod-A/main/data/fieldcare"
ASSET_FILES = [
    "fieldcare_manifest.json",
    "service_docs.jsonl",
    "equipment_records.csv",
    "maintenance_history.csv",
    "service_tickets.csv",
    "tool_schemas.json",
    "tool_fixture_responses.json",
    "user_requests.jsonl",
    "eval_cases.jsonl",
]


def load_openrouter_key(required: bool = False) -> Optional[str]:
    key = os.getenv("OPENROUTER_API_KEY")
    if not key:
        try:
            from google.colab import userdata

            key = userdata.get("OPENROUTER_API_KEY")
        except Exception:
            key = None
    if not key and required:
        key = getpass("OpenRouter API key: ").strip()
    if key:
        os.environ["OPENROUTER_API_KEY"] = key
        return key
    return None


def candidate_asset_dirs() -> list[Path]:
    dirs = [
        Path("fieldcare"),
        Path("assets/fieldcare"),
        Path("data/fieldcare"),
        Path("/content/fieldcare"),
        Path("/content/data/fieldcare"),
        Path("lessons/ml-app-dev/module-a/campus/sprint-4/assets/fieldcare"),
    ]
    return dirs


def read_asset_text(filename: str) -> tuple[str, str]:
    for directory in candidate_asset_dirs():
        path = directory / filename
        if path.exists():
            return path.read_text(encoding="utf-8"), str(path)

    url = f"{FIELDCARE_RAW_BASE_URL}/{filename}"
    request = Request(url, headers={"User-Agent": "fieldcare-sprint-4-colab"})
    try:
        with urlopen(request, timeout=30) as response:
            return response.read().decode("utf-8"), url
    except URLError as exc:
        raise RuntimeError(
            "Could not load FieldCare assets locally or from GitHub. "
            "If you are running this before publication, upload the assets/fieldcare folder to Colab."
        ) from exc


def load_json_asset(filename: str) -> tuple[dict[str, Any], str]:
    text, source = read_asset_text(filename)
    return json.loads(text), source


def load_jsonl_asset(filename: str) -> tuple[list[dict[str, Any]], str]:
    text, source = read_asset_text(filename)
    rows = [json.loads(line) for line in text.splitlines() if line.strip()]
    return rows, source


def load_csv_asset(filename: str) -> tuple[pd.DataFrame, str]:
    text, source = read_asset_text(filename)
    return pd.read_csv(StringIO(text), keep_default_na=False), source


def pretty(obj: Any) -> None:
    display(JSON(json.loads(json.dumps(obj, default=str, ensure_ascii=False))))


OPENROUTER_READY = load_openrouter_key(required=False) is not None
HELPER_CORE_READY = Document is not None and BM25Retriever is not None and ToolRegistry is not None
HELPER_CORE_STATUS = {
    "helper_core_available": HELPER_CORE_READY,
    "retrieval_api": "BM25Retriever" if BM25Retriever is not None else "local fallback",
    "tool_api": "ToolRegistry" if ToolRegistry is not None else "local fallback",
    "import_error": HELPER_CORE_IMPORT_ERROR,
}
display(JSON({"openrouter_key_available": OPENROUTER_READY, "value_hidden": OPENROUTER_READY, "helper_core": HELPER_CORE_STATUS}))


## 3. Load the FieldCare project environment

Load the same synthetic assets used by the Campus lessons and live sessions. Inspect the manifest before writing any application logic. The manifest tells you what each asset is for and which component consumes it.


In [ ]:
manifest, manifest_source = load_json_asset("fieldcare_manifest.json")
service_docs, docs_source = load_jsonl_asset("service_docs.jsonl")
equipment_df, equipment_source = load_csv_asset("equipment_records.csv")
maintenance_df, maintenance_source = load_csv_asset("maintenance_history.csv")
tickets_df, tickets_source = load_csv_asset("service_tickets.csv")
tool_schemas, tool_schema_source = load_json_asset("tool_schemas.json")
tool_fixtures, tool_fixture_source = load_json_asset("tool_fixture_responses.json")
user_requests, user_requests_source = load_jsonl_asset("user_requests.jsonl")
eval_cases, eval_cases_source = load_jsonl_asset("eval_cases.jsonl")

loaded_assets = pd.DataFrame(
    [
        {"asset": "fieldcare_manifest.json", "rows_or_items": len(manifest.get("assets", [])), "source": manifest_source},
        {"asset": "service_docs.jsonl", "rows_or_items": len(service_docs), "source": docs_source},
        {"asset": "equipment_records.csv", "rows_or_items": len(equipment_df), "source": equipment_source},
        {"asset": "maintenance_history.csv", "rows_or_items": len(maintenance_df), "source": maintenance_source},
        {"asset": "service_tickets.csv", "rows_or_items": len(tickets_df), "source": tickets_source},
        {"asset": "tool_schemas.json", "rows_or_items": len(tool_schemas["tools"]), "source": tool_schema_source},
        {"asset": "tool_fixture_responses.json", "rows_or_items": len(tool_fixtures["fixtures"]), "source": tool_fixture_source},
        {"asset": "user_requests.jsonl", "rows_or_items": len(user_requests), "source": user_requests_source},
        {"asset": "eval_cases.jsonl", "rows_or_items": len(eval_cases), "source": eval_cases_source},
    ]
)
display(loaded_assets)
display(Markdown(f"**Scenario**\n\n{manifest['scenario']}"))


## 4. Scope the application

Retrieval means selecting useful source text before a model writes an answer. A tool is application code with a defined input and output contract. MCP-style access means the capability behaves like an external tool boundary rather than hidden prompt text.

Complete the scope decisions before building. Use the request bank to decide which request types the application should support and which ones should ask, abstain, or escalate.


In [ ]:
requests_df = pd.DataFrame(user_requests)
display(requests_df[["request_id", "request_type", "difficulty", "equipment_id", "ticket_id", "expected_sources"]])

SUPPORTED_REQUEST_TYPES = [
    # TODO: choose the request types your FieldCare application will support.
    "troubleshooting_plus_warranty",
    "retrieval_only_airflow",
    "warranty_only",
    "ticket_status_only",
    "repeat_fault_escalation",
]

OUT_OF_SCOPE_REQUEST_TYPES = [
    # TODO: choose request types the app should abstain from or route.
    "unsupported_model",
    "unsupported_business_promise",
]

APPLICATION_SCOPE = {
    "user": "field-service technician",
    "workflow_moment": "diagnose equipment issue and decide whether to answer, ask, abstain, or escalate",
    "supported_request_types": SUPPORTED_REQUEST_TYPES,
    "out_of_scope_request_types": OUT_OF_SCOPE_REQUEST_TYPES,
    "retrieval_sources": ["service_docs.jsonl"],
    "tool_sources": [
        "get_equipment_record",
        "get_warranty_status",
        "get_maintenance_history",
        "get_ticket_status",
        "recommend_escalation_path",
    ],
    "must_ask_for_more_info_when": [
        "equipment_id or model is missing for model-specific repair guidance",
        "ticket closure is requested without a ticket_id",
    ],
    "must_escalate_when": [
        "safety indicators are present",
        "repeat fault threshold is met",
        "warranty data is unavailable for a coverage decision",
    ],
}

pretty(APPLICATION_SCOPE)


## 5. Specify inputs, outputs, and success criteria

A response contract is the shape the application should return. It keeps the application from producing a confident paragraph when evidence is missing. Edit the contract and criteria so they match the scope you chose.


In [ ]:
RESPONSE_CONTRACT = {
    "answer_summary": "short technician-facing answer",
    "next_checks": ["ordered diagnostic checks or empty list"],
    "coverage_statement": "covered | not_covered | conditional | cannot_verify | not_applicable",
    "evidence": [{"doc_id": "DOC-FC-...", "reason": "why this source matters"}],
    "tool_state": [{"tool_name": "get_...", "status": "found | not_found | unavailable | timeout"}],
    "decision_flags": ["cite_evidence", "ask_for_more_information", "abstain", "escalate"],
    "escalation_path": "team or null",
    "limitations": ["what the app cannot verify"],
}

SUCCESS_CRITERIA = [
    {"criterion": "Required current documents appear in the evidence set", "measure": "all required_doc_ids found unless the case asks for more information"},
    {"criterion": "Warranty answers use the warranty tool", "measure": "coverage questions include get_warranty_status in tool_state"},
    {"criterion": "Unsafe or unsupported requests do not receive routine repair instructions", "measure": "response includes abstain or escalate flag"},
    {"criterion": "Legacy or irrelevant docs do not drive the final answer", "measure": "must_not_use_doc_ids are absent from accepted evidence"},
]

pretty({"response_contract": RESPONSE_CONTRACT, "success_criteria": SUCCESS_CRITERIA})


## 6. Build retrieval over service documentation

The starter retriever uses `BM25Retriever` from `ms-ai-ml-helper-core` so the evidence path matches the retrieval helper used in earlier notebooks. Reranking means taking an initial candidate list and ordering it again using query-specific usefulness. Your job is to improve the reranker so the final context prefers current, model-appropriate, safety-critical, and policy-relevant evidence.


In [ ]:
TOKEN_RE = re.compile(r"[a-z0-9]+")


def tokenize(text: str) -> list[str]:
    return TOKEN_RE.findall(text.lower())


def doc_chunks(docs: list[dict[str, Any]]) -> list[dict[str, Any]]:
    chunks = []
    for doc in docs:
        paragraphs = [p.strip() for p in doc["text"].split("\n\n") if p.strip()]
        for index, paragraph in enumerate(paragraphs, start=1):
            chunks.append(
                {
                    "chunk_id": f"{doc['doc_id']}#C{index:02d}",
                    "doc_id": doc["doc_id"],
                    "title": doc["title"],
                    "doc_type": doc["doc_type"],
                    "version": doc["version"],
                    "effective_date": doc["effective_date"],
                    "text": paragraph,
                    "topics": doc["topics"],
                    "applies_to_models": doc["applies_to_models"],
                    "current": doc["current"],
                    "safety_level": doc["safety_level"],
                    "supersedes": doc.get("supersedes", []),
                }
            )
    return chunks


chunks = doc_chunks(service_docs)
display(pd.DataFrame(chunks)[["chunk_id", "doc_id", "title", "current", "safety_level"]].head(10))


def chunk_to_helper_document(chunk: dict[str, Any]):
    metadata = dict(chunk)
    searchable_text = "\n".join(
        [
            chunk["title"],
            "Topics: " + ", ".join(chunk["topics"]),
            "Models: " + ", ".join(chunk["applies_to_models"]),
            chunk["text"],
        ]
    )
    return Document(id=chunk["chunk_id"], text=searchable_text, metadata=metadata)


helper_documents = [chunk_to_helper_document(chunk) for chunk in chunks] if Document is not None else []
bm25_retriever = BM25Retriever.from_documents(helper_documents) if BM25Retriever is not None else None
display(JSON({"retrieval_backend": "ms-ai-ml-helper-core.BM25Retriever" if bm25_retriever else "local lexical fallback", "indexed_chunks": len(chunks)}))


def lexical_score(query: str, text: str) -> float:
    q = Counter(tokenize(query))
    d = Counter(tokenize(text))
    if not q or not d:
        return 0.0
    overlap = sum(min(q[token], d[token]) for token in q)
    return overlap / math.sqrt(sum(q.values()) * sum(d.values()))


def baseline_retrieve(query: str, top_k: int = 8) -> list[dict[str, Any]]:
    if bm25_retriever is not None:
        matches = bm25_retriever.search(query, top_k=top_k)
        max_score = max((match.keyword_score for match in matches), default=1.0)
        rows = []
        for match in matches:
            baseline_score = match.keyword_score / max_score if max_score else 0.0
            rows.append(
                {
                    **match.document.metadata,
                    "baseline_score": round(baseline_score, 4),
                    "raw_bm25_score": round(match.keyword_score, 4),
                    "retrieval_backend": "BM25Retriever",
                }
            )
        return rows

    rows = []
    for chunk in chunks:
        score = lexical_score(query, chunk["title"] + " " + chunk["text"] + " " + " ".join(chunk["topics"]))
        rows.append({**chunk, "baseline_score": round(score, 4), "retrieval_backend": "local_lexical_fallback"})
    return sorted(rows, key=lambda row: row["baseline_score"], reverse=True)[:top_k]


RERANK_CONFIG = {
    "prefer_current_docs": True,
    "model_match_boost": 0.12,
    "model_mismatch_penalty": 0.35,
    "product_note_model_boost": 0.2,
    "model_conflict_note_boost": 0.35,
    "topic_match_boost": 0.04,
    "safety_override_boost": 0.12,
    "warranty_policy_boost": 0.24,
    "filter_procedure_boost": 0.18,
    "overheat_triage_boost": 0.16,
    "sensor_workflow_boost": 0.18,
    "repeat_fault_policy_boost": 0.2,
    "superseded_current_doc_boost": 0.35,
    "legacy_penalty": 0.45,
}


def infer_model_from_request(request_text: str, equipment_record: Optional[dict[str, Any]] = None) -> Optional[str]:
    if equipment_record:
        return equipment_record.get("model")
    for model in ["HX-200", "HX-220", "VX-500", "AX-90"]:
        if model.lower() in request_text.lower():
            return model
    return None


def rerank_candidates(query: str, candidates: list[dict[str, Any]], equipment_record: Optional[dict[str, Any]] = None) -> list[dict[str, Any]]:
    model = infer_model_from_request(query, equipment_record)
    query_tokens = set(tokenize(query))
    reranked = []
    for row in candidates:
        score = row["baseline_score"]

        # TODO: adjust these rules as you learn which evidence should reach the final context.
        if RERANK_CONFIG["prefer_current_docs"] and not row["current"]:
            score -= RERANK_CONFIG["legacy_penalty"]
        if model and model in row["applies_to_models"]:
            score += RERANK_CONFIG["model_match_boost"]
        if model and row["applies_to_models"] and model not in row["applies_to_models"]:
            score -= RERANK_CONFIG["model_mismatch_penalty"]
        if model and row["doc_type"] == "product_note" and model in row["applies_to_models"]:
            score += RERANK_CONFIG["product_note_model_boost"]
        if model and row["doc_type"] == "product_note" and model in row["applies_to_models"] and any(term in query_tokens for term in ["vx", "hx", "model", "guidance", "trust"]):
            score += RERANK_CONFIG["model_conflict_note_boost"]
        if any(topic_token in query_tokens for topic in row["topics"] for topic_token in tokenize(topic)):
            score += RERANK_CONFIG["topic_match_boost"]
        if row["safety_level"] == "critical" and any(term in query_tokens for term in ["burning", "smoke", "scorched", "breaker"]):
            score += RERANK_CONFIG["safety_override_boost"]
        if row["doc_id"] == "DOC-FC-WAR-004" and any(term in query_tokens for term in ["warranty", "covered", "coverage", "invoice", "pay"]):
            score += RERANK_CONFIG["warranty_policy_boost"]
        if row["doc_id"] == "DOC-FC-MP-014" and any(term in query_tokens for term in ["filter", "filters", "replacement", "replace", "airflow", "dusty", "parts"]):
            score += RERANK_CONFIG["filter_procedure_boost"]
        if row["doc_id"] == "DOC-FC-TS-001" and any(term in query_tokens for term in ["overheating", "overheat", "hot", "e", "117"]):
            score += RERANK_CONFIG["overheat_triage_boost"]
        if row["doc_id"] == "DOC-FC-TS-009" and any(term in query_tokens for term in ["sensor", "sensors", "calibration", "drift", "e", "221"]):
            score += RERANK_CONFIG["sensor_workflow_boost"]
        if row["doc_id"] == "DOC-FC-ESC-007" and any(term in query_tokens for term in ["again", "repeat", "recurring", "callback", "third", "escalate"]):
            score += RERANK_CONFIG["repeat_fault_policy_boost"]
        if row["doc_id"] in ["DOC-FC-TS-001", "DOC-FC-MP-014"] and any(term in query_tokens for term in ["2019", "legacy", "bulletin", "reset", "superseded"]):
            score += RERANK_CONFIG["superseded_current_doc_boost"]

        reranked.append({**row, "rerank_score": round(score, 4)})
    return sorted(reranked, key=lambda row: row["rerank_score"], reverse=True)


def show_evidence(rows: list[dict[str, Any]], score_column: str = "rerank_score", columns: Optional[list[str]] = None) -> None:
    if columns is None:
        columns = ["chunk_id", "doc_id", "title", "current", "safety_level", score_column, "text"]
    display(pd.DataFrame(rows)[columns])


focus_request = next(row for row in user_requests if row["request_id"] == "REQ-FC-001")
baseline_candidates = baseline_retrieve(focus_request["request_text"], top_k=8)
reranked_context = rerank_candidates(focus_request["request_text"], baseline_candidates)[:5]

display(Markdown("### Baseline candidates"))
show_evidence(baseline_candidates, "baseline_score", ["chunk_id", "doc_id", "title", "current", "baseline_score"])
display(Markdown("### Reranked context"))
show_evidence(reranked_context, "rerank_score", ["chunk_id", "doc_id", "title", "current", "rerank_score"])


## 7. Add the FieldCare tool layer

The tool layer simulates MCP-style structured access: the application calls named functions with validated inputs and receives structured status objects. The model should not have to guess equipment records, warranty state, or current ticket state from retrieved documents.


In [ ]:
equipment_by_id = {row["equipment_id"]: row for row in equipment_df.to_dict("records")}
tickets_by_id = {row["ticket_id"]: row for row in tickets_df.to_dict("records")}
maintenance_records = maintenance_df.to_dict("records")


def tool_response(status: str, data: Any, message: str, error_type: Optional[str], source_id: str) -> dict[str, Any]:
    return {
        "status": status,
        "data": data,
        "message": message,
        "error_type": error_type,
        "source_id": source_id,
        "retrieved_at": SIMULATED_CURRENT_DATE,
    }


def get_equipment_record(equipment_id: str) -> dict[str, Any]:
    if not equipment_id:
        return tool_response("invalid_input", None, "equipment_id is required.", "missing_argument", "equipment_records.csv")
    record = equipment_by_id.get(equipment_id)
    if not record:
        return tool_response("not_found", None, f"No equipment record found for {equipment_id}.", "missing_record", "equipment_records.csv")
    return tool_response("found", record, f"Equipment record found for {equipment_id}.", None, "equipment_records.csv")


def get_ticket_status(ticket_id: str) -> dict[str, Any]:
    if not ticket_id:
        return tool_response("invalid_input", None, "ticket_id is required.", "missing_argument", "service_tickets.csv")
    record = tickets_by_id.get(ticket_id)
    if not record:
        return tool_response("not_found", None, f"No ticket found for {ticket_id}.", "missing_record", "service_tickets.csv")
    return tool_response("found", record, f"Ticket record found for {ticket_id}.", None, "service_tickets.csv")


def get_maintenance_history(equipment_id: str, limit: int = 5, issue_category: Optional[str] = None) -> dict[str, Any]:
    if not equipment_id:
        return tool_response("invalid_input", None, "equipment_id is required.", "missing_argument", "maintenance_history.csv")
    rows = [row for row in maintenance_records if row["equipment_id"] == equipment_id]
    if issue_category:
        issue = issue_category.lower()
        rows = [row for row in rows if issue in row["observed_issue"].lower() or issue in row["visit_type"].lower()]
    rows = sorted(rows, key=lambda row: row["service_date"], reverse=True)[:limit]
    status = "found" if rows else "not_found"
    return tool_response(status, rows, f"{len(rows)} maintenance record(s) found.", None if rows else "empty_result", "maintenance_history.csv")


def get_warranty_status(equipment_id: str, service_date: Optional[str] = None) -> dict[str, Any]:
    if equipment_id == "EQ-FC-TIMEOUT":
        return tool_response("timeout", None, "Warranty service did not respond inside the client timeout.", "timeout", "warranty_service_fixture")
    equipment = equipment_by_id.get(equipment_id)
    if not equipment:
        return tool_response("not_found", None, f"No equipment record found for {equipment_id}.", "missing_record", "warranty_service_fixture")
    status = equipment["warranty_status"]
    if equipment_id == "EQ-FC-1006":
        return tool_response("unavailable", None, "Warranty data is unavailable during asset migration.", "upstream_unavailable", "warranty_service_fixture")
    if status == "active":
        data = {
            "coverage_state": "active",
            "parts_coverage": "active",
            "labor_coverage": "active_when_policy_conditions_match",
            "exclusions": [],
            "confidence": "high",
            "service_date": service_date or SIMULATED_CURRENT_DATE,
        }
    elif status == "active_limited":
        data = {
            "coverage_state": "active_limited",
            "parts_coverage": "active",
            "labor_coverage": "excluded_pending_review",
            "exclusions": ["unapproved_relocation"],
            "confidence": "medium",
            "service_date": service_date or SIMULATED_CURRENT_DATE,
        }
    elif status == "expired":
        data = {
            "coverage_state": "expired",
            "parts_coverage": "expired",
            "labor_coverage": "expired",
            "exclusions": ["warranty_period_ended"],
            "confidence": "high",
            "service_date": service_date or SIMULATED_CURRENT_DATE,
        }
    else:
        data = {
            "coverage_state": status,
            "parts_coverage": "unknown",
            "labor_coverage": "unknown",
            "exclusions": ["unsupported_or_unknown_status"],
            "confidence": "low",
            "service_date": service_date or SIMULATED_CURRENT_DATE,
        }
    return tool_response("found", data, f"Warranty state for {equipment_id}: {data['coverage_state']}.", None, "warranty_service_fixture")


def recommend_escalation_path(equipment_id: str, ticket_id: str, reason_code: str) -> dict[str, Any]:
    if not equipment_id or not ticket_id:
        return tool_response("invalid_input", None, "equipment_id and ticket_id are required for routing.", "missing_argument", "routing_fixture")
    team_by_reason = {
        "safety_indicator": "Safety Engineering",
        "repeat_fault": "Field Engineering",
        "unsupported_model": "Legacy Service Desk",
        "warranty_unavailable": "Warranty Operations",
        "permission_failure": "Authorized Account Owner",
    }
    team = team_by_reason.get(reason_code)
    if not team:
        return tool_response("invalid_input", None, f"Unsupported reason_code: {reason_code}.", "invalid_reason_code", "routing_fixture")
    data = {
        "recommended_team": team,
        "reason_code": reason_code,
        "ticket_id": ticket_id,
        "equipment_id": equipment_id,
        "note": "Recommendation only. No ticket action was performed.",
    }
    return tool_response("found", data, f"Route to {team}.", None, "routing_fixture")


def search_service_docs(query: str, model: Optional[str] = None, topics: Optional[list[str]] = None, top_k: int = 5) -> dict[str, Any]:
    query_with_topics = " ".join([query, " ".join(topics or [])]).strip()
    candidates = baseline_retrieve(query_with_topics, top_k=min(max(top_k * 2, top_k), 10))
    equipment_record = {"model": model} if model else None
    rows = rerank_candidates(query, candidates, equipment_record=equipment_record)[:top_k]
    data = [
        {
            "doc_id": row["doc_id"],
            "chunk_id": row["chunk_id"],
            "title": row["title"],
            "score": row["rerank_score"],
            "current": row["current"],
            "preview": row["text"][:220] + ("..." if len(row["text"]) > 220 else ""),
        }
        for row in rows
    ]
    status = "found" if data else "not_found"
    return tool_response(status, data, f"{len(data)} documentation chunk(s) found.", None if data else "empty_result", "service_docs.jsonl")


TOOLBOX = {
    "get_equipment_record": get_equipment_record,
    "get_warranty_status": get_warranty_status,
    "get_maintenance_history": get_maintenance_history,
    "get_ticket_status": get_ticket_status,
    "search_service_docs": search_service_docs,
    "recommend_escalation_path": recommend_escalation_path,
}

fieldcare_tool_registry = ToolRegistry() if ToolRegistry is not None else None
registered_tool_names = []
if fieldcare_tool_registry is not None:
    for schema in tool_schemas["tools"]:
        handler = TOOLBOX.get(schema["name"])
        if handler is None:
            continue
        fieldcare_tool_registry.register(
            name=schema["name"],
            description=schema["description"],
            parameters=schema["parameters"],
            handler=handler,
        )
        registered_tool_names.append(schema["name"])

openrouter_tool_definitions = fieldcare_tool_registry.to_openrouter_tools() if fieldcare_tool_registry is not None else []
pretty(
    {
        "available_tools": list(TOOLBOX),
        "schema_count": len(tool_schemas["tools"]),
        "registered_with_helper_tool_registry": registered_tool_names,
        "openrouter_tool_definition_count": len(openrouter_tool_definitions),
    }
)


## 8. Assemble context and run an end-to-end slice

Orchestration means choosing the order of retrieval, tool calls, validation, and response drafting. Start small: one request, visible evidence, one context object, one final response object. Then improve the weak parts.


In [ ]:
ID_RE = re.compile(r"(EQ-FC-[A-Z0-9]+|TCK-FC-[0-9]+)")


def extract_known_ids(text: str) -> dict[str, Optional[str]]:
    values = ID_RE.findall(text)
    equipment_id = next((value for value in values if value.startswith("EQ-FC-")), None)
    ticket_id = next((value for value in values if value.startswith("TCK-FC-")), None)
    return {"equipment_id": equipment_id, "ticket_id": ticket_id}


TOOL_PLAN_BY_REQUEST_TYPE = {
    "troubleshooting_plus_warranty": ["get_equipment_record", "get_maintenance_history", "get_warranty_status", "get_ticket_status", "recommend_escalation_path"],
    "retrieval_only_airflow": [],
    "warranty_only": ["get_equipment_record", "get_warranty_status", "get_ticket_status"],
    "sensor_plus_warranty": ["get_equipment_record", "get_maintenance_history", "get_warranty_status"],
    "expired_warranty": ["get_equipment_record", "get_warranty_status", "get_maintenance_history"],
    "tool_unavailable": ["get_equipment_record", "get_warranty_status", "get_ticket_status", "recommend_escalation_path"],
    "conflicting_documentation": ["get_equipment_record", "get_ticket_status"],
    "reranking_distractor": ["get_equipment_record", "get_ticket_status"],
    "ticket_status_only": ["get_ticket_status"],
    "repeat_fault_escalation": ["get_equipment_record", "get_maintenance_history", "get_ticket_status", "recommend_escalation_path"],
    "safety_escalation": ["get_equipment_record", "get_ticket_status", "recommend_escalation_path"],
    "unsupported_model": ["get_equipment_record", "get_ticket_status", "recommend_escalation_path"],
    "unsupported_business_promise": ["get_warranty_status"],
}


RETRIEVAL_REQUIRED_BY_REQUEST_TYPE = {
    "incomplete_request": False,
    "insufficient_information": False,
    "ticket_status_only": False,
    "unsupported_model": False,
}


def should_retrieve(request: dict[str, Any]) -> bool:
    return RETRIEVAL_REQUIRED_BY_REQUEST_TYPE.get(request["request_type"], True)


def request_ids_for(request: dict[str, Any]) -> dict[str, str]:
    extracted = extract_known_ids(request["request_text"])
    return {
        "equipment_id": extracted["equipment_id"] or request.get("equipment_id") or "",
        "ticket_id": extracted["ticket_id"] or request.get("ticket_id") or "",
    }


def infer_escalation_reason(request: dict[str, Any], tool_results: list[dict[str, Any]]) -> Optional[str]:
    text = request["request_text"].lower()
    request_type = request["request_type"]
    if any(term in text for term in ["burning", "smoke", "scorched", "breaker"]):
        return "safety_indicator"
    if any(term in text for term in ["permission", "cannot access", "not authorized"]):
        return "permission_failure"
    if any(row["tool_name"] == "get_warranty_status" and row["result"]["status"] in ["unavailable", "timeout", "permission_denied"] for row in tool_results):
        return "warranty_unavailable"
    equipment = next((row["result"]["data"] for row in tool_results if row["tool_name"] == "get_equipment_record" and row["result"]["status"] == "found"), None)
    if equipment and equipment.get("warranty_status") == "unsupported_model":
        return "unsupported_model"
    if request_type in ["repeat_fault_escalation", "troubleshooting_plus_warranty"] or any(term in text for term in ["again", "repeat", "recurring", "callback", "third"]):
        return "repeat_fault"
    return None


def build_tool_args(tool_name: str, request: dict[str, Any], tool_results: list[dict[str, Any]]) -> Optional[dict[str, Any]]:
    ids = request_ids_for(request)
    equipment_id = ids["equipment_id"]
    ticket_id = ids["ticket_id"]
    if tool_name == "get_equipment_record":
        return {"equipment_id": equipment_id}
    if tool_name == "get_warranty_status":
        return {"equipment_id": equipment_id, "service_date": SIMULATED_CURRENT_DATE}
    if tool_name == "get_maintenance_history":
        return {"equipment_id": equipment_id, "limit": 5}
    if tool_name == "get_ticket_status":
        return {"ticket_id": ticket_id}
    if tool_name == "recommend_escalation_path":
        reason_code = infer_escalation_reason(request, tool_results)
        if reason_code is None:
            return None
        return {"equipment_id": equipment_id, "ticket_id": ticket_id, "reason_code": reason_code}
    return None


def execute_registered_tool(tool_name: str, args: dict[str, Any], call_index: int) -> dict[str, Any]:
    if fieldcare_tool_registry is None:
        result = TOOLBOX[tool_name](**args)
        return {"tool_name": tool_name, "args": args, "result": result, "registry_validated": False}

    tool_call = {
        "id": f"fieldcare-tool-call-{call_index:02d}",
        "function": {"name": tool_name, "arguments": json.dumps(args, ensure_ascii=False)},
    }
    execution = fieldcare_tool_registry.execute_tool_call(tool_call)
    try:
        payload = json.loads(execution.content)
    except json.JSONDecodeError:
        payload = {"raw_content": execution.content}
    if execution.ok:
        result = payload
    else:
        result = tool_response(
            "invalid_input",
            None,
            payload.get("message", "Tool execution failed."),
            payload.get("error_type", execution.error_type),
            "ms-ai-ml-helper-core.ToolRegistry",
        )
    return {
        "tool_name": execution.name or tool_name,
        "args": args,
        "result": result,
        "registry_validated": True,
        "tool_call_id": execution.tool_call_id,
        "retryable": execution.retryable,
    }


def execute_tool_plan(request: dict[str, Any]) -> list[dict[str, Any]]:
    results = []
    for tool_name in TOOL_PLAN_BY_REQUEST_TYPE.get(request["request_type"], []):
        args = build_tool_args(tool_name, request, results)
        if args is None:
            continue
        results.append(execute_registered_tool(tool_name, args, len(results) + 1))
    return results


def accepted_evidence(query: str, tool_results: list[dict[str, Any]], top_k: int = 5) -> list[dict[str, Any]]:
    equipment_result = next((row["result"] for row in tool_results if row["tool_name"] == "get_equipment_record" and row["result"]["status"] == "found"), None)
    equipment_record = equipment_result["data"] if equipment_result else None
    model = infer_model_from_request(query, equipment_record)
    search_query = f"{query} {model or ''}".strip()
    candidates = baseline_retrieve(search_query, top_k=len(chunks))
    ranked = rerank_candidates(query, candidates, equipment_record=equipment_record)
    accepted = []
    seen_doc_ids = set()
    for row in ranked:
        if not row["current"]:
            continue
        if model and row["applies_to_models"] and model not in row["applies_to_models"]:
            continue
        if row["doc_id"] in seen_doc_ids:
            continue
        accepted.append(row)
        seen_doc_ids.add(row["doc_id"])
        if len(accepted) >= top_k:
            break
    return accepted


def classify_flags(request: dict[str, Any], evidence: list[dict[str, Any]], tool_results: list[dict[str, Any]]) -> list[str]:
    flags = set()
    text = request["request_text"].lower()
    request_type = request["request_type"]
    ids = request_ids_for(request)
    if request_type in ["incomplete_request", "insufficient_information"] or (not ids["equipment_id"] and any(term in text for term in ["unit", "close", "covered", "warranty"])):
        flags.update(["ask_for_more_information", "abstain"])
    if request_type == "ticket_status_only":
        flags.add("ticket_state_only")
    if request_type == "unsupported_business_promise" or ("promise" in text and "invoice" in text):
        flags.update(["business_boundary", "mention_coverage_condition", "abstain"])
    if any(term in text for term in ["warranty", "covered", "coverage", "invoice"]):
        flags.add("mention_coverage_condition")
    if any(term in text for term in ["permission", "cannot access", "not authorized"]):
        flags.update(["permission_or_workflow_boundary", "escalate"])
    if any(term in text for term in ["burning", "smoke", "scorched", "breaker"]):
        flags.update(["safety_escalation", "abstain", "escalate"])
    if any(row["result"]["status"] in ["unavailable", "timeout", "permission_denied"] for row in tool_results):
        flags.update(["mention_uncertainty", "abstain", "escalate"])
    equipment = next((row["result"]["data"] for row in tool_results if row["tool_name"] == "get_equipment_record" and row["result"]["status"] == "found"), None)
    if equipment and equipment.get("warranty_status") == "unsupported_model":
        flags.update(["unsupported_model", "abstain", "escalate"])
    maintenance = next((row["result"]["data"] for row in tool_results if row["tool_name"] == "get_maintenance_history" and row["result"]["status"] == "found"), [])
    if maintenance and (request_type in ["sensor_plus_warranty", "expired_warranty"] or any(term in text for term in ["again", "third", "callback", "replacement"])):
        flags.add("history_changes_answer")
    if request_type == "reranking_distractor":
        flags.add("reject_irrelevant_evidence")
    if request_type == "conflicting_documentation":
        flags.add("reject_outdated_evidence")
    if any(row["doc_id"] == "DOC-FC-ESC-007" for row in evidence) and request["request_type"] in ["repeat_fault_escalation", "troubleshooting_plus_warranty"]:
        flags.add("escalate")
    if any(row["tool_name"] == "recommend_escalation_path" and row["result"]["status"] == "found" for row in tool_results):
        flags.add("escalate")
    if evidence:
        flags.add("cite_evidence")
    return sorted(flags)


def assemble_context(request: dict[str, Any]) -> dict[str, Any]:
    tool_results = execute_tool_plan(request)
    evidence = accepted_evidence(request["request_text"], tool_results) if should_retrieve(request) else []
    flags = classify_flags(request, evidence, tool_results)
    return {
        "request": request,
        "retrieved_evidence": [
            {
                "chunk_id": row["chunk_id"],
                "doc_id": row["doc_id"],
                "title": row["title"],
                "current": row["current"],
                "score": row["rerank_score"],
                "preview": row["text"][:260] + ("..." if len(row["text"]) > 260 else ""),
            }
            for row in evidence
        ],
        "tool_results": tool_results,
        "decision_flags": flags,
    }


def draft_starter_response(context: dict[str, Any]) -> dict[str, Any]:
    request = context["request"]
    flags = context["decision_flags"]
    tool_state = [
        {"tool_name": row["tool_name"], "status": row["result"]["status"], "message": row["result"]["message"], "registry_validated": row.get("registry_validated", False)}
        for row in context["tool_results"]
    ]
    evidence = [{"doc_id": row["doc_id"], "chunk_id": row["chunk_id"], "title": row["title"]} for row in context["retrieved_evidence"][:4]]
    warranty_result = next((row["result"] for row in context["tool_results"] if row["tool_name"] == "get_warranty_status"), None)
    escalation_result = next((row["result"] for row in context["tool_results"] if row["tool_name"] == "recommend_escalation_path" and row["result"]["status"] == "found"), None)

    # TODO: replace this starter assembly with your own response policy or optional model call.
    if "safety_escalation" in flags:
        summary = "Stop routine troubleshooting and escalate to Safety Engineering before repair guidance."
    elif "unsupported_model" in flags:
        summary = "FieldCare should not guide this repair because the model is outside support scope."
    elif "ask_for_more_information" in flags and not evidence:
        summary = "Ask for the equipment ID, ticket ID, model, error codes, and current readings before advising."
    elif "mention_uncertainty" in flags:
        summary = "The application cannot verify one required tool result, so it should route before making a confident claim."
    elif "escalate" in flags:
        summary = "Continue with targeted checks, but escalate because repeat-fault or ticket-state evidence requires review."
    else:
        summary = "Use the retrieved FieldCare evidence and available tool state to answer within scope."

    if warranty_result and warranty_result["status"] in ["unavailable", "timeout", "permission_denied"]:
        coverage_statement = "cannot_verify"
    elif warranty_result and warranty_result["status"] == "found" and warranty_result["data"].get("coverage_state") == "expired":
        coverage_statement = "not_covered"
    elif "mention_coverage_condition" in flags:
        coverage_statement = "conditional"
    else:
        coverage_statement = "not_applicable"

    if escalation_result:
        escalation_path = escalation_result["data"]["recommended_team"]
    elif "permission_or_workflow_boundary" in flags or (warranty_result and warranty_result["status"] in ["unavailable", "timeout", "permission_denied"]):
        escalation_path = "Warranty Operations"
    elif "unsupported_model" in flags:
        escalation_path = "Legacy Service Desk"
    elif "safety_escalation" in flags:
        escalation_path = "Safety Engineering"
    elif "escalate" in flags:
        escalation_path = "Field Engineering or owning operations team"
    else:
        escalation_path = None

    return {
        "request_id": request["request_id"],
        "answer_summary": summary,
        "next_checks": ["Inspect accepted evidence and tool state before writing the final technician response."],
        "coverage_statement": coverage_statement,
        "evidence": evidence,
        "tool_state": tool_state,
        "decision_flags": flags,
        "escalation_path": escalation_path,
        "limitations": ["Starter response is scaffolded; learners should refine response rules and model prompt."],
    }


def run_fieldcare_app(request_id: str) -> dict[str, Any]:
    request = next(row for row in user_requests if row["request_id"] == request_id)
    context = assemble_context(request)
    response = draft_starter_response(context)
    return {"context": context, "response": response}


run_result = run_fieldcare_app("REQ-FC-001")
pretty(run_result["response"])
display(pd.DataFrame(run_result["context"]["retrieved_evidence"])[["doc_id", "chunk_id", "title", "current", "score"]])


## 9. Optional live model response

Use this only when instructed and when your OpenRouter key is available. Keep the same context object. The model may draft wording, but the application still owns retrieval, tools, validation, abstention, and escalation flags.


In [ ]:
RUN_LIVE_MODEL = False
MODEL_ID = "google/gemini-3.1-flash-lite"


def generate_model_response(context: dict[str, Any]) -> str:
    if not RUN_LIVE_MODEL:
        return "Live model call skipped. Set RUN_LIVE_MODEL = True only when instructed and when OPENROUTER_API_KEY is available."
    if OpenRouterClient is None:
        return "OpenRouterClient is unavailable in this runtime."
    if not load_openrouter_key(required=False):
        return "OPENROUTER_API_KEY is not available."

    client = OpenRouterClient(app_title="fieldcare-sprint-4-project")
    messages = [
        {
            "role": "system",
            "content": (
                "You are a HelioDesk FieldCare assistant. Answer only from the supplied context. "
                "Preserve decision_flags. If a flag says ask, abstain, or escalate, do that visibly. "
                "Cite doc_id values and tool statuses."
            ),
        },
        {"role": "user", "content": json.dumps(context, ensure_ascii=False)},
    ]
    response = client.chat(messages, model=MODEL_ID, temperature=0, max_tokens=700)
    return response.content or ""


display(Markdown(generate_model_response(run_result["context"])))


## 10. Evaluate expected behavior

Evaluation compares what the application actually did against the reusable test set. This is not a perfect grader. It is a practical lens that exposes missing evidence, missing tools, unsafe confidence, and weak boundaries.


In [ ]:
def request_for_eval_case(eval_case: dict[str, Any]) -> dict[str, Any]:
    request = dict(next(row for row in user_requests if row["request_id"] == eval_case["request_id"]))
    if eval_case["input"] != request["request_text"]:
        request["request_text"] = eval_case["input"]
        ids = extract_known_ids(eval_case["input"])
        if ids["equipment_id"] is not None:
            request["equipment_id"] = ids["equipment_id"]
        if ids["ticket_id"] is not None:
            request["ticket_id"] = ids["ticket_id"]
    return request


def evaluate_case(eval_case: dict[str, Any]) -> dict[str, Any]:
    request = request_for_eval_case(eval_case)
    result = {"context": assemble_context(request)}
    result["response"] = draft_starter_response(result["context"])

    actual_doc_ids = {row["doc_id"] for row in result["context"]["retrieved_evidence"]}
    actual_tools = {row["tool_name"] for row in result["context"]["tool_results"]}
    actual_flags = set(result["response"]["decision_flags"])

    required_docs = set(eval_case["required_doc_ids"])
    required_tools = set(eval_case["required_tool_calls"])
    required_flags = set(eval_case["expected_response_flags"])
    forbidden_docs = set(eval_case["must_not_use_doc_ids"])

    return {
        "eval_id": eval_case["eval_id"],
        "request_id": eval_case["request_id"],
        "missing_required_docs": sorted(required_docs - actual_doc_ids),
        "forbidden_docs_present": sorted(forbidden_docs & actual_doc_ids),
        "missing_required_tools": sorted(required_tools - actual_tools),
        "missing_expected_flags": sorted(required_flags - actual_flags),
        "actual_doc_ids": sorted(actual_doc_ids),
        "actual_tools": sorted(actual_tools),
        "actual_flags": sorted(actual_flags),
        "starter_pass": not (required_docs - actual_doc_ids or forbidden_docs & actual_doc_ids or required_tools - actual_tools or required_flags - actual_flags),
    }


evaluation_results = [evaluate_case(case) for case in eval_cases]
evaluation_df = pd.DataFrame(evaluation_results)
display(evaluation_df[["eval_id", "request_id", "starter_pass", "missing_required_docs", "forbidden_docs_present", "missing_required_tools", "missing_expected_flags"]])


## 11. Improve one weak behavior and rerun

Pick one failing row from the evaluation table. Change exactly one of these surfaces, then rerun the affected cells:

- `RERANK_CONFIG`
- `TOOL_PLAN_BY_REQUEST_TYPE`
- `classify_flags(...)`
- `draft_starter_response(...)`

Record the before-and-after evidence. Do not tune against all cases at once; one targeted improvement is easier to explain.


In [ ]:
TARGET_EVAL_ID = "EVAL-FC-001"  # TODO: choose one failing or high-priority evaluation case.
target_case = next(case for case in eval_cases if case["eval_id"] == TARGET_EVAL_ID)
before_after_note = {
    "target_eval_id": TARGET_EVAL_ID,
    "weak_behavior_observed": "TODO: describe what failed in the evaluation table.",
    "surface_changed": "TODO: name the one function or setting changed.",
    "before_evidence": "TODO: paste doc IDs, tool statuses, or flags before the change.",
    "after_evidence": "TODO: paste doc IDs, tool statuses, or flags after rerun.",
    "remaining_risk": "TODO: name one thing still not production-ready.",
}
pretty(before_after_note)


## 12. Edge-case log

Edge cases are product states that reveal how the application behaves when information is missing, contradictory, failed, unsafe, or outside scope. Run at least four cases from `eval_cases.jsonl`, then fill the log.


In [ ]:
EDGE_CASE_LOG_COLUMNS = [
    "eval_id",
    "input",
    "expected_behavior",
    "actual_behavior",
    "retrieved_evidence",
    "tool_information",
    "observed_failure",
    "likely_cause",
    "possible_improvement",
]

edge_case_log = pd.DataFrame(columns=EDGE_CASE_LOG_COLUMNS)
edge_case_log.loc[len(edge_case_log)] = {
    "eval_id": "EVAL-FC-006",
    "input": next(case for case in eval_cases if case["eval_id"] == "EVAL-FC-006")["input"],
    "expected_behavior": "Warranty unavailable -> route rather than invent coverage.",
    "actual_behavior": "TODO: run the app and summarize.",
    "retrieved_evidence": "TODO: list accepted doc IDs.",
    "tool_information": "TODO: list tool statuses.",
    "observed_failure": "TODO: describe mismatch or write none.",
    "likely_cause": "TODO: retrieval, tool plan, classification, prompt, or response assembly.",
    "possible_improvement": "TODO: one targeted change.",
}
display(edge_case_log)


## 13. FieldCare readiness assessment

Finish by deciding what the application can honestly claim. A readiness assessment is not a celebration note. It is a compact account of what works, what fails, what evidence supports the claim, and what would need to change before production use.


In [ ]:
FIELDCARE_READINESS_ASSESSMENT = {
    "works_reliably": [
        "TODO: name one request family that passes with evidence."
    ],
    "still_fails": [
        "TODO: name one failing request family and cite the evaluation row."
    ],
    "limitations": [
        "starter retrieval is lexical and local",
        "model response is optional and not required for deterministic evaluation",
        "tool layer is a fixture, not a production backend",
    ],
    "meets_success_criteria": "TODO: yes, partly, or no. Cite SUCCESS_CRITERIA and evaluation evidence.",
    "before_production": [
        "replace fixture tools with authorized services",
        "add monitoring and logs",
        "expand evaluation set with real field data after privacy review",
        "confirm warranty and escalation owners",
    ],
}

pretty(FIELDCARE_READINESS_ASSESSMENT)
